<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINT-Colab/blob/main/notebooks/05_qwen35_visao_geoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 05 · Pistas de GEOINT em imagens com Qwen3.5 4B (visão)

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINT-Colab)

O Qwen3.5 é um modelo unificado de texto e visão. Aqui ele **descreve pistas** de localização (idioma de placas, arquitetura, vegetação, clima, veículos) em JSON estruturado. Ele não dá a resposta final: as pistas guiam o analista, que confirma com fontes como mapas e imagens de satélite. Também extraímos o EXIF com Pillow.

| | |
|---|---|
| **Hardware** | GPU T4 gratuita (Ambiente de execução → Alterar tipo → T4 GPU) |
| **Tempo estimado** | 10 minutos |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

In [ ]:
%%capture
# Instalação igual à dos notebooks oficiais da Unsloth (LGPL-3.0), mantida para compatibilidade.
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINT-Colab.git /content/OSINT-Colab 2>/dev/null || git -C /content/OSINT-Colab pull -q
import sys
sys.path.insert(0, "/content/OSINT-Colab")
from osintia import auditoria, calibracao, coleta, coordenacao, custodia, dados, esquemas, extracao, geo, roteamento
print("OSINTIA pronto")

In [ ]:
from unsloth import FastVisionModel
MODELO = "unsloth/Qwen3.5-4B"
model, tokenizer = FastVisionModel.from_pretrained(MODELO, load_in_4bit=True)
FastVisionModel.for_inference(model)

## 1. Envie uma imagem
Use uma imagem que você tem direito de analisar. Fotos enviadas para redes sociais costumam perder o EXIF; arquivos originais não.

In [ ]:
from google.colab import files
from PIL import Image
enviado = files.upload()
caminho = next(iter(enviado))
imagem = Image.open(caminho).convert("RGB")
imagem.thumbnail((1024, 1024))
imagem

## 2. EXIF (metadados)

In [ ]:
from PIL.ExifTags import TAGS, GPSTAGS
bruto = Image.open(caminho).getexif()
exif = {TAGS.get(k, k): v for k, v in bruto.items()}
gps = {GPSTAGS.get(k, k): v for k, v in bruto.get_ifd(0x8825).items()} if bruto else {}
print({k: exif[k] for k in ("Make", "Model", "DateTime", "Software") if k in exif})
print("GPS:", gps or "sem GPS no arquivo")

## 3. Pistas estruturadas

In [ ]:
import json, re
from transformers import TextStreamer

PEDIDO = """Analise a imagem como um analista de GEOINT. Não chute a localização final.
Responda só com JSON neste formato:
{"idioma_textos": [], "textos_legiveis": [], "arquitetura": "", "vegetacao": "", "clima_estacao": "",
 "veiculos_e_mao_de_direcao": "", "infraestrutura": "", "paises_candidatos": [], "proximos_passos": []}"""

msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PEDIDO}]}]
texto = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
entradas = tokenizer(imagem, texto, add_special_tokens=False, return_tensors="pt").to("cuda")
saida = model.generate(**entradas, max_new_tokens=700, use_cache=True, temperature=0.3, min_p=0.1)
resposta = tokenizer.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True)
blocos = re.findall(r"\{.*\}", resposta, flags=re.S)
pistas = json.loads(blocos[-1]) if blocos else {"bruto": resposta}
print(json.dumps(pistas, indent=2, ensure_ascii=False))

### Leitura
- Textos legíveis e idioma são as pistas mais fortes; vegetação e arquitetura apenas restringem.
- `paises_candidatos` é hipótese, não conclusão. Confirme cada uma.
- Modelos de visão podem **inventar texto** que não está na imagem: confira os `textos_legiveis` olhando a foto.

In [ ]:
import hashlib, os
os.makedirs("/content/saidas", exist_ok=True)
with open("/content/saidas/geoint.json", "w", encoding="utf-8") as f:
    json.dump({"arquivo": caminho, "exif": {k: str(v) for k, v in exif.items()}, "gps": {k: str(v) for k, v in gps.items()},
               "pistas": pistas, "modelo": MODELO}, f, ensure_ascii=False, indent=2)
custodia.registrar("/content/saidas/manifesto.jsonl", "coleta_imagem", caminho)
custodia.registrar("/content/saidas/manifesto.jsonl", "pistas_geoint", "/content/saidas/geoint.json", {"modelo": MODELO})

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- `docs/COLABS-RELACIONADOS.md` lista outros notebooks de OSINT (Bellingcat e comunidade) que combinam com este.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*